# ML-05: Feature Vector and Leakage/Privacy Check

This notebook uses the data contract from ML-04. One row is one eligible pseudonymized client-content item at the March 31, 2026 cutoff.

## 1. Build the feature vector

The five features cover March 2 through March 31, 2026. The decline proxy covers April 1 through April 30, 2026.

In [1]:
from huggingface_hub import get_token
import duckdb
import numpy as np
import pandas as pd

HF_TOKEN = get_token()
con = duckdb.connect()
con.execute("CREATE OR REPLACE SECRET hf (TYPE huggingface, TOKEN ?)", [HF_TOKEN])

REL = "hf://datasets/FlyRank/internship-warehouse"
MARCH = f"read_parquet('{REL}/fact_content_daily_performance/month=2026-03/*.parquet')"
APRIL = f"read_parquet('{REL}/fact_content_daily_performance/month=2026-04/*.parquet')"
CUTOFF_DATE = "2026-03-31"
FEATURE_START = "2026-03-02"
TARGET_START = "2026-04-01"
TARGET_END = "2026-04-30"
MIN_FEATURE_IMPRESSIONS = 100
DECLINE_RATIO = 0.80

feature_frame = con.sql(f"""
    WITH feature_page AS (
        SELECT
            client_hash_id,
            content_hash_id,
            SUM(gsc_impressions) AS impressions_feature_30d,
            SUM(gsc_clicks) AS clicks_feature_30d,
            SUM(gsc_sum_position) AS position_sum_feature_30d,
            SUM(gsc_impressions) FILTER (
                WHERE report_date <= DATE '{CUTOFF_DATE}' - INTERVAL 15 DAY
            ) AS impressions_early15,
            SUM(gsc_impressions) FILTER (
                WHERE report_date > DATE '{CUTOFF_DATE}' - INTERVAL 15 DAY
            ) AS impressions_late15
        FROM {MARCH}
        WHERE gsc_data_available IS TRUE
          AND report_date BETWEEN DATE '{FEATURE_START}' AND DATE '{CUTOFF_DATE}'
        GROUP BY 1, 2
        HAVING COUNT(DISTINCT report_date) = 30
    ),
    target_page AS (
        SELECT
            client_hash_id,
            content_hash_id,
            SUM(gsc_impressions) AS impressions_target_30d
        FROM {APRIL}
        WHERE gsc_data_available IS TRUE
          AND report_date BETWEEN DATE '{TARGET_START}' AND DATE '{TARGET_END}'
        GROUP BY 1, 2
        HAVING COUNT(DISTINCT report_date) = 30
    )
    SELECT
        f.client_hash_id,
        f.content_hash_id,
        DATE '{CUTOFF_DATE}' AS cutoff_date,
        f.impressions_feature_30d,
        f.clicks_feature_30d,
        100.0 * f.clicks_feature_30d
            / NULLIF(f.impressions_feature_30d, 0) AS ctr_feature_30d_pct,
        f.position_sum_feature_30d
            / NULLIF(f.impressions_feature_30d, 0) AS weighted_position_feature_30d,
        100.0 * (f.impressions_late15 - f.impressions_early15)
            / NULLIF(f.impressions_early15, 0)
            AS impressions_change_late15_vs_early15_pct,
        t.impressions_target_30d,
        CASE
            WHEN t.impressions_target_30d < {DECLINE_RATIO} * f.impressions_feature_30d THEN 1
            ELSE 0
        END AS is_declining_label
    FROM feature_page f
    JOIN target_page t USING (client_hash_id, content_hash_id)
    WHERE f.impressions_feature_30d >= {MIN_FEATURE_IMPRESSIONS}
    ORDER BY f.client_hash_id, f.content_hash_id
""").df()

FEATURES = [
    "impressions_feature_30d",
    "clicks_feature_30d",
    "ctr_feature_30d_pct",
    "weighted_position_feature_30d",
    "impressions_change_late15_vs_early15_pct",
]
LABEL = "is_declining_label"

X = feature_frame[FEATURES].copy()
y = feature_frame[LABEL].copy()

print(f"Feature frame: {len(X):,} eligible client-page rows")
print(f"Observed decline-proxy rate: {y.mean():.1%}")
display(feature_frame[FEATURES + [LABEL]].head())

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

Feature frame: 50,641 eligible client-page rows
Observed decline-proxy rate: 47.3%


,impressions_feature_30d,clicks_feature_30d,ctr_feature_30d_pct,weighted_position_feature_30d,impressions_change_late15_vs_early15_pct,is_declining_label
0,325.0,2.0,0.615385,14.529231,64.227642,0
1,446.0,0.0,0.000000,14.273543,-39.568345,0
2,746.0,5.0,0.670241,12.805630,-41.276596,0
3,3214.0,19.0,0.591164,7.054449,40.749064,0
4,2021.0,1.0,0.049480,53.063335,-50.738552,1


## 2. Feature notes (meaning, missing, categorical, available when?)

All five features are numeric and available after GSC reporting through March 31 is complete.

| Feature | Meaning | Missing handling |
|---|---|---|
| `impressions_feature_30d` | Total GSC impressions in the 30-day feature window | No fill |
| `clicks_feature_30d` | Total GSC clicks in the 30-day feature window | No fill. Zero clicks are valid. |
| `ctr_feature_30d_pct` | Clicks divided by impressions, multiplied by 100 | No fill. Eligible rows have at least 100 impressions. |
| `weighted_position_feature_30d` | Impression-weighted GSC position | No fill. Eligible rows have at least 100 impressions. |
| `impressions_change_late15_vs_early15_pct` | Impression change from the first 15 days to the last 15 days | Leave blank. Drop before modeling. |

In [2]:
missing_by_feature = X.isna().sum().rename("missing_rows").to_frame()
display(missing_by_feature)
print(f"Rows with at least one missing feature: {X.isna().any(axis=1).sum():,}")

,missing_rows
impressions_feature_30d,0
clicks_feature_30d,0
ctr_feature_30d_pct,0
weighted_position_feature_30d,0
impressions_change_late15_vs_early15_pct,0


Rows with at least one missing feature: 0


## 3. The leakage hunt

The feature window ends on March 31 and the label window starts on April 1. I also compare the five features with a deliberately leaked field based on April impressions.

In [3]:
from sklearn.model_selection import train_test_split
from sklearn.tree import DecisionTreeClassifier

feature_window_end = pd.Timestamp(CUTOFF_DATE)
label_window_start = pd.Timestamp(TARGET_START)
print("Feature window ends before label window:", feature_window_end < label_window_start)

LEAKED_FEATURE = "future_impressions_change_pct"
model_frame = feature_frame.dropna(subset=FEATURES + [LABEL]).copy()
model_frame[LEAKED_FEATURE] = (
    100.0 * (model_frame["impressions_target_30d"] - model_frame["impressions_feature_30d"])
    / model_frame["impressions_feature_30d"]
)

train_rows, test_rows = train_test_split(
    model_frame.index,
    test_size=0.25,
    random_state=42,
    stratify=model_frame[LABEL],
)

def score(columns):
    tree = DecisionTreeClassifier(max_depth=3, class_weight="balanced", random_state=42)
    tree.fit(model_frame.loc[train_rows, columns], model_frame.loc[train_rows, LABEL])
    probabilities = tree.predict_proba(model_frame.loc[test_rows, columns])[:, 1]
    y_test = model_frame.loc[test_rows, LABEL]
    top_50 = np.argsort(probabilities)[-50:]
    return y_test.iloc[top_50].mean()

feature_p50 = score(FEATURES)
leaky_p50 = score(FEATURES + [LEAKED_FEATURE])

print(f"Five-feature tree Precision@50: {feature_p50:.3f}")
print(f"Tree with future field Precision@50: {leaky_p50:.3f}")

del model_frame[LEAKED_FEATURE]
print(f"Removed leaked column: {LEAKED_FEATURE}")

Feature window ends before label window: True
Five-feature tree Precision@50: 0.860
Tree with future field Precision@50: 1.000
Removed leaked column: future_impressions_change_pct


The future field uses April impressions, which also define the label. In this run, it raised Precision@50 from 0.860 to 1.000. I removed it after the check.

## 4. What I excluded and why

| Fields | Reason |
|---|---|
| `client_hash_id`, `content_hash_id`, `cutoff_date` | Used for grouping, joins, and checking dates |
| `gsc_data_available` | Used to require complete GSC coverage |
| `impressions_target_30d`, `is_declining_label`, `future_impressions_change_pct` | Future values and the label |
| GA4, session, AI, and scroll fields | Availability is limited and unavailable rows contain zero-filled measurements |
| `health_score`, `priority_score`, `action_type` | Not in this table. Product outputs could contain the answer. |

In [4]:
print("Final feature list:", FEATURES)

Final feature list: ['impressions_feature_30d', 'clicks_feature_30d', 'ctr_feature_30d_pct', 'weighted_position_feature_30d', 'impressions_change_late15_vs_early15_pct']


## Self-check

Before you submit, confirm each line honestly:

- [X] Every section above is filled — markdown thinking AND the code that backs it
- [X] The notebook runs top to bottom with no errors (Runtime → Run all)
- [X] No client names, URLs, or private queries anywhere
- [X] My claims use careful words: observed, measured, directional, decision-support
- [X] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.